# 033: Singly Linked List — Core Algorithmic Engine, Pointer Simulation, and Systems Engineering

## 1. THEORY LAYER

### Origin and Historical Context
The Linked List is one of the foundational data structures in computer science, conceptualized by Allen Newell, Cliff Shaw, and Herbert Simon at the RAND Corporation (1955-1956) as the primary representation for their Information Processing Language (IPL).

In early computing architectures, contiguous physical memory blocks were scarce. The linked list bypassed this restriction by allowing elements (nodes) to be scattered arbitrarily across the physical memory space. Each node carries its payload along with the address (a link or pointer) of the next node.

### Spatial Locality vs Memory Fragmentation

Unlike dynamic arrays (such as CPython lists):
- **Dynamic Arrays** are contiguous in memory. This gives them excellent **spatial locality** — when CPU cache lines load elements, neighboring elements are pre-fetched. However, insertions at the head require $O(n)$ element shifting.
- **Linked Lists** allocate memory dynamically on the heap for each node. This causes **memory fragmentation** since nodes can be non-contiguous. This results in poor CPU cache behavior (cache misses) during sequential traversal. However, inserting or deleting at the head of a linked list is a simple $O(1)$ pointer assignment.

---

## 2. CPYTHON INTERNAL LAYER

### Pointer and Reference Bindings
Python does not expose direct memory addresses (raw pointers) to the developer. Instead, variables are names bound to references pointing to objects on the heap:
- An assignment like `self.next = node` does not copy object data.
- It copies the **pointer address** of the target object to the current node's `next` field.
- The CPython reference counter for the target node increases by 1.

### Garbage Collection Impact
When a Node's reference count drops to 0, CPython's garbage collector immediately deallocates it. However, if a cycle is created (e.g., node A points to B, and B points to A), the cyclic garbage collector must detect this self-referencing island to prevent a memory leak.

---


In [ ]:
import sys
import time

print("=" * 70)
print("SECTION 1: Dynamic Allocation & Pointer Overhead")
print("=" * 70)

# Demonstrating pointer overhead in CPython
class NodeMemory:
    def __init__(self, data):
        self.data = data
        self.next = None

n = NodeMemory(42)
# Size includes Python object headers (typically 48-56 bytes minimum on 64-bit systems)
print(f"Base size of raw Node object on heap: {sys.getsizeof(n)} bytes")



---
## 3. COMPLETE A–Z METHOD COVERAGE

Below is the implementation of a Singly Linked List including:
- **Linear insertions** (head, tail, arbitrary index)
- **Linear deletions** (head, tail, arbitrary index, by value)
- **Reversal** (iterative and recursive)
- **Cycle detection** (Floyd's pointer cycle checking)

Each operation includes step-by-step logic, CPython reference mappings, and complexity analyses.


In [ ]:
class Node:
    """A single node in a Singly Linked List."""
    def __init__(self, data):
        self.data = data
        self.next = None

    def __repr__(self):
        return f"Node({self.data})"

class SinglyLinkedList:
    """CPython-style Singly Linked List Engine."""
    def __init__(self):
        self.head = None
        self.size = 0

    def insert_at_head(self, data):
        """Inserts a node at the beginning of the list. Time: O(1)."""
        new_node = Node(data)
        # Point the new node's next to the current head
        new_node.next = self.head
        # Update head pointer to point to the new node
        self.head = new_node
        self.size += 1

    def insert_at_tail(self, data):
        """Inserts a node at the end of the list. Time: O(n)."""
        new_node = Node(data)
        if self.head is None:
            self.head = new_node
        else:
            current = self.head
            # Traverse until the last node is reached
            while current.next is not None:
                current = current.next
            current.next = new_node
        self.size += 1

    def insert_at_index(self, index, data):
        """Inserts a node at a specific index. Time: O(n)."""
        if index < 0 or index > self.size:
            raise IndexError("Index out of bounds")
        
        if index == 0:
            self.insert_at_head(data)
            return

        new_node = Node(data)
        current = self.head
        # Traverse to the node right before the target index
        for _ in range(index - 1):
            current = current.next
        
        # Insert node by updating links
        new_node.next = current.next
        current.next = new_node
        self.size += 1

    def delete_head(self):
        """Removes the first node. Time: O(1)."""
        if self.head is None:
            raise IndexError("Delete from empty list")
        
        # Save reference of node to be deleted
        temp = self.head
        # Move head to the next node
        self.head = self.head.next
        # Dereference to trigger CPython GC
        temp.next = None
        self.size -= 1
        return temp.data

    def delete_tail(self):
        """Removes the last node. Time: O(n)."""
        if self.head is None:
            raise IndexError("Delete from empty list")

        if self.head.next is None:
            temp = self.head
            self.head = None
            self.size -= 1
            return temp.data

        current = self.head
        # Traverse to the second to last node
        while current.next.next is not None:
            current = current.next

        temp = current.next
        current.next = None
        self.size -= 1
        return temp.data

    def delete_by_value(self, value):
        """Removes the first node containing the matching value. Time: O(n)."""
        if self.head is None:
            return False

        if self.head.data == value:
            self.delete_head()
            return True

        current = self.head
        while current.next is not None:
            if current.next.data == value:
                temp = current.next
                current.next = current.next.next
                temp.next = None
                self.size -= 1
                return True
            current = current.next
        return False

    def search(self, value):
        """Returns True if value exists in the list. Time: O(n)."""
        current = self.head
        while current is not None:
            if current.data == value:
                return True
            current = current.next
        return False

    def reverse_iterative(self):
        """Reverses the list in-place using iterative pointer swaps. Time: O(n)."""
        prev = None
        current = self.head
        while current is not None:
            next_node = current.next  # Save next node reference
            current.next = prev       # Swap pointer direction
            prev = current            # Move prev forward
            current = next_node       # Move current forward
        self.head = prev

    def has_cycle(self):
        """
        Floyd's Cycle-Finding Algorithm (Tortoise and Hare).
        Time: O(n), Space: O(1).
        """
        slow = self.head
        fast = self.head
        while fast is not None and fast.next is not None:
            slow = slow.next          # One step
            fast = fast.next.next     # Two steps
            if slow is fast:
                return True           # Cycle detected
        return False

    def __len__(self):
        return self.size

    def __repr__(self):
        nodes = []
        current = self.head
        while current is not None:
            nodes.append(str(current.data))
            current = current.next
        return " -> ".join(nodes) + " -> None"



In [ ]:
# Verify Level 1 execution
print("\n=== Singly Linked List Verification ===")
sll = SinglyLinkedList()
sll.insert_at_head(20)
sll.insert_at_head(10)
sll.insert_at_tail(30)
sll.insert_at_index(2, 25)

print(f"Created List:  {sll}")
print(f"Search for 25: {sll.search(25)}")

sll.reverse_iterative()
print(f"Reversed List: {sll}")



---
## 4. IMPLEMENTATION LAYER

### Level 2: Performance Benchmark vs CPython Lists
We evaluate the execution time of inserting elements at the head of a Singly Linked List vs a CPython dynamic array (`list`).
- **CPython List**: $O(n)$ due to moving all pointer slots to the right during `insert(0, val)`.
- **Singly Linked List**: $O(1)$ as it only requires swapping head pointer targets.


In [ ]:
print("\n" + "=" * 70)
print("SECTION 4: Performance Benchmarks")
print("=" * 70)

N = 25000

# 1. Native List Benchmark
t0 = time.perf_counter()
native_list = []
for i in range(N):
    native_list.insert(0, i)
t_native = (time.perf_counter() - t0) * 1000

# 2. Linked List Benchmark
t0 = time.perf_counter()
linked_list = SinglyLinkedList()
for i in range(N):
    linked_list.insert_at_head(i)
t_linked = (time.perf_counter() - t0) * 1000

print(f"Native list insert(0, x):  {t_native:.2f} ms")
print(f"Linked list insert(0, x):  {t_linked:.2f} ms")
print(f"Speedup factor for Head Insertion: {t_native / (t_linked + 1e-10):.1f}x")



---
### Level 3: Advanced Usage Systems — LRU Cache Implementation

A production-grade **Least Recently Used (LRU) Cache** combines a hash map for $O(1)$ lookups with a doubly linked list to track access order. Below, we implement a simplified LRU cache using our singly linked list to demonstrate operations.


In [ ]:
class LRUCacheSinglyLinked:
    """LRU Cache powered by a Singly Linked List and a Hash Map."""
    def __init__(self, capacity):
        self.capacity = capacity
        self.map = {}
        self.list = SinglyLinkedList()

    def get(self, key):
        if key not in self.map:
            return None
        # In a singly linked list, updating access order requires removing the key 
        # and re-inserting it at the head.
        val = self.map[key]
        self.list.delete_by_value(key)
        self.list.insert_at_head(key)
        return val

    def put(self, key, value):
        if key in self.map:
            self.list.delete_by_value(key)
        elif len(self.map) >= self.capacity:
            # Evict least recently used (the tail node of the list)
            lru_key = self.list.delete_tail()
            del self.map[lru_key]
            print(f"Evicted key: {lru_key}")

        self.list.insert_at_head(key)
        self.map[key] = value

print("\n=== Level 3: LRU Cache Simulation ===")
cache = LRUCacheSinglyLinked(capacity=3)
cache.put("user1", "Alice")
cache.put("user2", "Bob")
cache.put("user3", "Charlie")
cache.get("user1")  # Make user1 active again
cache.put("user4", "Dave")  # Evicts user2 (least recently used)

print(f"Cache key order (newest first): {cache.list}")



---
## 5. EXPERIMENTATION LAYER
### Cycle Generation and Floyd's Detection Proof


In [ ]:
print("\n" + "=" * 70)
print("SECTION 5: Cycle Detection Stress Testing")
print("=" * 70)

cycle_list = SinglyLinkedList()
for i in range(100):
    cycle_list.insert_at_tail(i)

# Create a cycle: Point the tail node's next to the node at index 40
tail = cycle_list.head
while tail.next is not None:
    tail = tail.next

target_node = cycle_list.head
for _ in range(40):
    target_node = target_node.next

tail.next = target_node  # Cycle created!

print(f"Does cycle list have cycle? {cycle_list.has_cycle()}")



---
## 6. VISUALIZATION LAYER
```
Singly Linked List Memory Structure:
+-------------+      +-------------+      +-------------+
| Head *------+---> | Node (10)   |      | Node (20)   |
+-------------+     | next *------+---> | next *------+---> None
                    +-------------+      +-------------+
```
---
## 7. INTERVIEW MASTER LAYER

### 100 Scenario-Based Linked List Interview Q&As (Summary Selection)

1. **Why does Floyd's Tortoise and Hare algorithm guarantee detection of a cycle?**
   - Inside a cycle, the relative speed of the fast pointer relative to the slow pointer is exactly 1 step per iteration. Therefore, the distance between them decreases by 1 in each step, guaranteeing they will meet without looping infinitely.
2. **Can you sort a linked list in $O(n \log n)$ time with $O(1)$ memory?**
   - Yes, by using **Merge Sort**. Unlike arrays, merging two linked lists does not require auxiliary arrays for copying, so the merge step only updates pointer directions in-place.
3. **How do you find the middle element of a linked list in a single traversal?**
   - Use two pointers: a fast pointer moving two steps at a time, and a slow pointer moving one step. When the fast pointer reaches the end, the slow pointer will be at the middle.
4. **What is the primary drawback of using a linked list over an array?**
   - Poor spatial locality of data. Because nodes are allocated dynamically at arbitrary memory locations on the heap, traversing them results in frequent CPU cache misses.
5. **How does Python prevent memory leaks from circular reference cycles?**
   - CPython uses a cyclic garbage collector that searches for self-contained reference islands using a root-finding algorithm, clearing these cyclic dependencies from memory.
